# Edge user allocation on GAViST5G: full Python pipeline

Run the cells top to bottom with a TensorFlow kernel. Each step calls `eua_pipeline.py` in a separate process, so GPU memory is released after every step and the output appears live below the cell.

* Open this notebook from the `python` folder (next to `eua_pipeline.py`).
* Put the two GAViST5G CSV files in `../data` first (see `../data/README.md`).
* `train` skips model/seed combinations that already finished, so you can rerun any cell after an interruption.

In [ ]:
# 1. Environment check
import sys, os, subprocess
print("Python:", sys.executable)
assert os.path.exists("eua_pipeline.py"), "Open this notebook from the python folder (next to eua_pipeline.py)"
import tensorflow as tf
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
# helper: run one step with live output (stops the notebook if the step fails)
PY = sys.executable
def run(script, *args, cpu_only=False):
    env = os.environ.copy()
    if cpu_only:
        env["CUDA_VISIBLE_DEVICES"] = "-1"
    cmd = [PY, "-u", script, *map(str, args)]
    print(">>", " ".join(cmd[2:]))
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"step failed (exit code {p.returncode}); see the output above")

DATA   = os.path.join("..", "data", "dataset1.csv")       # Canadian portion
GLOBAL = os.path.join("..", "data", "dataset2.csv")   # international portion
SEEDS = [0, 1, 2, 3, 4]

## 2. Prepare data (about 30 s)

In [ ]:
run("eua_pipeline.py", "prepare", "--data", DATA, "--work", "work")
run("eua_pipeline.py", "prepare", "--data", GLOBAL, "--single-app", "YouTube", "--work", "work_global")

## 3. Smoke test: every model, 1 epoch, 6 users (a few minutes)

In [ ]:
run("eua_pipeline.py", "smoke", "--work", "work", "--out", "results")

## 4. Baselines (seconds)

In [ ]:
run("eua_pipeline.py", "baselines", "--work", "work", "--out", "results")
run("eua_pipeline.py", "baselines", "--work", "work_global", "--out", "results_global")

## 5. Optional: equal-budget Bayesian optimisation (several hours)
Uncomment to run. Afterwards set `TUNED = ["--use-tuned"]` so that training and latency use the selected hyperparameters.

In [ ]:
# run("eua_pipeline.py", "tune", "--work", "work", "--out", "results", "--models", "all", "--trials", 15)
TUNED = []

## 6. Seven DL models, 5 seeds each (the longest step)
Add `"--verbose"` to see per-epoch progress.

In [ ]:
run("eua_pipeline.py", "train", "--work", "work", "--out", "results", "--models", "all",
    "--feature-sets", "full", "--seeds", *SEEDS, *TUNED)

## 7. Feature ablation: without RTT, and without any server history

In [ ]:
run("eua_pipeline.py", "train", "--work", "work", "--out", "results", "--models", "DNN", "GRU", "Transformer", "FEDformer",
    "--feature-sets", "no_rtt", "no_srv", "--seeds", *SEEDS, *TUNED)

## 8. Unseen users: 5 user-disjoint folds

In [ ]:
run("eua_pipeline.py", "holdout", "--work", "work", "--out", "results", "--models", "GRU", "DNN", *TUNED)

## 9. International portion

In [ ]:
run("eua_pipeline.py", "train", "--work", "work_global", "--out", "results_global", "--models", "all",
    "--feature-sets", "full", "--seeds", 0, 1, 2, *TUNED)

## 10. Inference latency: GPU, then CPU only

In [ ]:
run("eua_pipeline.py", "latency", "--work", "work", "--out", "results", *TUNED)
run("eua_pipeline.py", "latency", "--work", "work", "--out", "results", *TUNED, cpu_only=True)

## 11. Reports

In [ ]:
run("eua_pipeline.py", "report", "--out", "results")
run("eua_pipeline.py", "report", "--out", "results_global")
from IPython.display import Markdown, display
display(Markdown(open(os.path.join("results", "REPORT.md"), encoding="utf-8").read()))

## 12. Inputs of the ns-3 evaluation
Writes `../ns3/ns3_inputs/`. Continue with `../ns3/README.md` on a Linux machine with ns-3.

In [ ]:
run(os.path.join("..", "ns3", "make_ns3_inputs.py"), "--python-dir", ".", "--out", os.path.join("..", "ns3", "ns3_inputs"))